# CLIP ViT-L/14 Feature Extraction and Classification

这个 Notebook 演示如何使用 `transformers` 里的经典 `CLIP ViT-L/14` 模型完成下面这条流程：

1. 读取一张图片
2. 提取图片的 `image feature`
3. 构造候选类别文本并提取 `text feature`
4. 用图文特征相似度直接预测“这张图最像哪一类”

这里重点不是微调，而是使用 CLIP 的零样本分类能力。

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch transformers pillow matplotlib
```

In [ ]:
# dataclass 用于统一管理实验配置
from dataclasses import dataclass
from pathlib import Path

# matplotlib 用于显示图片和预测结果
import matplotlib.pyplot as plt
# PIL 用于读取本地图片
from PIL import Image
# PyTorch 核心模块
import torch
import torch.nn.functional as F
# transformers 中的 CLIPModel 和 CLIPProcessor
from transformers import CLIPModel, CLIPProcessor

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 这里使用经典的 OpenAI CLIP ViT-L/14
    model_id: str = 'openai/clip-vit-large-patch14'
    # 这里填要预测的图片路径，默认留作占位示例
    image_path: str = './example.jpg'
    # top-k 展示多少个候选类别
    top_k: int = 5


cfg = Config()
cfg

## 2. 加载 CLIP 模型

这里使用 `transformers` 提供的 `CLIPModel` 和 `CLIPProcessor`。模型第一次运行时会自动下载权重。

In [ ]:
# processor 负责把图片和文本整理成模型可接受的输入格式
processor = CLIPProcessor.from_pretrained(cfg.model_id)
# model 负责提取图像特征和文本特征
model = CLIPModel.from_pretrained(cfg.model_id).to(device)
model.eval()

## 3. 读取图片

这里默认从本地路径读取一张图片。你可以把 `cfg.image_path` 改成你自己的文件路径。

In [ ]:
def load_image(image_path: str):
    # 检查文件是否存在，避免路径写错时直接报一个不清楚的错误
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'找不到图片文件: {image_path.resolve()}')
    # CLIP 期望 RGB 图片输入
    image = Image.open(image_path).convert('RGB')
    return image


image = load_image(cfg.image_path)
image.size

In [ ]:
# 可视化当前要预测的图片
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title('Input image')
plt.axis('off')
plt.show()

## 4. 定义候选类别文本

CLIP 的“直接预测”本质上是：
- 把图片编码成向量
- 把候选类别文本也编码成向量
- 计算相似度，谁最像就判成哪一类

这里先给一组 `CIFAR-10` 风格的示例类别。你也可以换成自己的类别列表。

In [ ]:
# 英文类别名更贴近 CLIP 原始训练分布，通常效果会更稳定
candidate_labels_en = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck',
]

# 这里保留一份中文显示名，便于展示结果
candidate_labels_zh = {
    'airplane': '飞机',
    'automobile': '汽车',
    'bird': '鸟',
    'cat': '猫',
    'deer': '鹿',
    'dog': '狗',
    'frog': '青蛙',
    'horse': '马',
    'ship': '船',
    'truck': '卡车',
}

# CLIP 对 prompt 很敏感，常用写法是 a photo of a ...
text_prompts = [f'a photo of a {label}' for label in candidate_labels_en]
text_prompts

## 5. 提取 image feature

下面这一步只做一件事：把图片编码成一个高维向量。这个向量就是后面分类判断的基础。

In [ ]:
@torch.no_grad()
def extract_image_feature(model, processor, image, device):
    # processor 会自动完成 resize、normalize 等模型需要的预处理
    image_inputs = processor(images=image, return_tensors='pt')
    image_inputs = {k: v.to(device) for k, v in image_inputs.items()}
    # 直接调用 get_image_features 提取图像向量
    image_features = model.get_image_features(**image_inputs)
    # 做 L2 归一化，方便后面直接用余弦相似度比较
    image_features = F.normalize(image_features, p=2, dim=-1)
    return image_features


image_features = extract_image_feature(model, processor, image, device)
print('image feature shape:', tuple(image_features.shape))
image_features[0, :10]

## 6. 提取 text feature

这一步把所有候选类别文本编码成向量。后面分类时，本质上就是比较图像向量和这些文本向量谁更接近。

In [ ]:
@torch.no_grad()
def extract_text_features(model, processor, text_prompts, device):
    # processor 会自动完成 tokenization 和 padding
    text_inputs = processor(text=text_prompts, return_tensors='pt', padding=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    # 直接调用 get_text_features 提取文本向量
    text_features = model.get_text_features(**text_inputs)
    # 做 L2 归一化，保证后面相似度比较更稳定
    text_features = F.normalize(text_features, p=2, dim=-1)
    return text_features


text_features = extract_text_features(model, processor, text_prompts, device)
print('text feature shape:', tuple(text_features.shape))
text_features[0, :10]

## 7. 用 feature 直接做预测

这里就是你要的核心部分：
- 不做额外分类头
- 不做微调
- 直接拿 `image feature` 和 `text feature` 做相似度
- 相似度最高的类别就是预测结果

In [ ]:
@torch.no_grad()
def predict_with_clip_features(image_features, text_features, labels_en, labels_zh, top_k=5):
    # 由于前面已经归一化，这里矩阵乘法就等价于余弦相似度
    similarity = image_features @ text_features.T
    # 乘一个温度系数，让 softmax 分布更有区分度
    logits = similarity * 100.0
    probs = logits.softmax(dim=-1)

    top_probs, top_indices = probs[0].topk(top_k)

    results = []
    for prob, idx in zip(top_probs.tolist(), top_indices.tolist()):
        label_en = labels_en[idx]
        label_zh = labels_zh.get(label_en, label_en)
        results.append({
            'label_en': label_en,
            'label_zh': label_zh,
            'prob': prob,
        })
    return results, similarity, probs


results, similarity, probs = predict_with_clip_features(
    image_features=image_features,
    text_features=text_features,
    labels_en=candidate_labels_en,
    labels_zh=candidate_labels_zh,
    top_k=cfg.top_k,
)

results

In [ ]:
# 打印 top-k 预测结果
for rank, item in enumerate(results, start=1):
    print(f"Top {rank}: {item['label_en']} / {item['label_zh']} -> {item['prob']:.4f}")

print('\n最终预测类别:')
print(f"{results[0]['label_en']} / {results[0]['label_zh']}")

In [ ]:
# 把 top-k 结果画成条形图，更直观看看分类置信度分布
labels = [f"{item['label_en']} / {item['label_zh']}" for item in results]
scores = [item['prob'] for item in results]

plt.figure(figsize=(10, 4))
plt.bar(labels, scores)
plt.title('CLIP zero-shot prediction scores')
plt.ylabel('Probability')
plt.xticks(rotation=20)
plt.show()

## 8. 单独看 feature 提取结果

如果你只关心 feature 本身，而不是分类，可以直接保留下面两个张量：
- `image_features`
- `text_features`

它们可以直接用于：
- 图像检索
- 图文相似度匹配
- 零样本分类
- 下游聚类或检索任务

In [ ]:
# image_features 是 1 张图片对应的特征向量
print('image_features.shape =', tuple(image_features.shape))
# text_features 是所有候选类别文本对应的特征向量
print('text_features.shape =', tuple(text_features.shape))

# 如果你只想拿 feature 去做别的任务，直接保存这两个变量即可
image_features, text_features

## 9. 为什么 CLIP 可以直接预测类别？

核心原因是 CLIP 在预训练时就学会了把图像和文本映射到同一个语义空间里。

这意味着：
- 如果图片是狗，那么图片向量应该更接近“a photo of a dog”这类文本向量
- 如果图片是飞机，那么图片向量应该更接近“a photo of an airplane”这类文本向量

所以推理阶段不一定需要再训练一个专门的分类头，直接比较图文特征相似度就可以完成零样本分类。

## 10. 可继续扩展的方向

- 把候选类别从 `CIFAR-10` 扩展到你自己的任务标签
- 对比不同 prompt 写法对预测结果的影响
- 一次输入多张图片，做批量 feature 提取和分类
- 把提取出来的 image feature 保存到向量库，用于检索任务